# Gold: tabelas especializadas por pergunta de negócio

Lê a Silver consolidada e gera 7 tabelas Gold, cada uma já agregada e pronta
para consulta direta via Athena, sem precisar repetir `GROUP BY` complexos a
cada consulta (ver `notebooks/05_analytics_athena/*.sql` para as queries
finais sobre estas tabelas). Lógica idêntica à validada em
`src/gold_builder.py` (pandas).

In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5


In [ ]:
%%configure
{
  "--enable-glue-datacatalog": "true"
}


In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
import pyspark.sql.functions as F
import pyspark.sql.types as T

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


In [ ]:
# =============================================================================
# CONFIGURACAO: valores conferidos ao vivo no console AWS do time em
# 2026-10-18 (Glue Data Catalog + S3). Troque so se os nomes mudarem.
#
# Os 3 CSVs brutos ficam juntos, soltos (sem subpasta por edicao), direto em
# s3://tech-challenge-gt26/bronze/, com nomes de arquivo que nao indicam a
# edicao de forma confiavel. O mapeamento abaixo foi confirmado por inspecao
# de conteudo (S3 Select: formato do cabecalho + pergunta "passou por layoff
# em <ano>"):
#   - "State_of_data_2022.csv"                      -> edicao 2023-2024
#   - "State_of_data_BR_2023_Kaggle - df_survey_2023.csv" -> edicao 2024-2025
#   - "Final Dataset - State of Data 2025-2026 - Kaggle.csv" -> edicao 2025-2026
#
# Os nomes de tabela abaixo (tb_bronze_*, tb_silver_*, tb_gold_*) usam o
# prefixo "tb_" de proposito, para conviver no mesmo database
# db_techchallenge_gt_fiap com as tabelas "bronze"/"silver"/"gold" que ja
# existem. Nada aqui sobrescreve ou apaga o que ja foi feito.
# =============================================================================
BUCKET = "s3://tech-challenge-gt26"

RAW_PATHS = {
    "2023-2024": f"{BUCKET}/bronze/State_of_data_2022.csv",
    "2024-2025": f"{BUCKET}/bronze/State_of_data_BR_2023_Kaggle - df_survey_2023.csv",
    "2025-2026": f"{BUCKET}/bronze/Final Dataset - State of Data 2025-2026 - Kaggle.csv",
}

# Prefixo novo e distinto do "bronze/" (lowercase) onde os CSVs brutos ja
# estao, para evitar qualquer risco de sobrescrever a fonte original.
BRONZE_PATH = f"{BUCKET}/bronze_parquet/"
SILVER_PATH = f"{BUCKET}/Silver/"
GOLD_PATH = f"{BUCKET}/Gold/"

DB_BRONZE = "db_techchallenge_gt_fiap"
DB_SILVER = "db_techchallenge_gt_fiap"
DB_GOLD = "db_techchallenge_gt_fiap"

for db in (DB_BRONZE, DB_SILVER, DB_GOLD):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")


In [ ]:
respondentes = spark.table(f"{DB_SILVER}.tb_silver_respondentes")
tecnologias = spark.table(f"{DB_SILVER}.tb_silver_tecnologias_adocao")

def gravar_gold(df, nome_tabela):
    (
        df.write.mode("overwrite").format("parquet")
        .option("path", GOLD_PATH + nome_tabela + "/")
        .saveAsTable(f"{DB_GOLD}.{nome_tabela}")
    )
    print(f"gravado e catalogado: {DB_GOLD}.{nome_tabela}  ({df.count():,} linhas)")


## gold_perfil_mercado: Q1 (estrutura do mercado)

In [ ]:
gold_perfil_mercado = (
    respondentes.filter(F.col("cargo_atual").isNotNull())
    .groupBy("ano_pesquisa", "regiao", "cargo_atual", "nivel_senioridade")
    .agg(F.count("*").alias("n_respondentes"))
)
gravar_gold(gold_perfil_mercado, "gold_perfil_mercado")


## gold_remuneracao: Q2 (perfis valorizados), Q3 (gap de gênero), Q6 (região/modelo)

In [ ]:
gold_remuneracao = (
    respondentes.filter(F.col("faixa_salarial_valor_medio").isNotNull())
    .groupBy("ano_pesquisa", "cargo_atual", "nivel_senioridade", "regiao", "genero", "modelo_trabalho_atual")
    .agg(
        F.mean("faixa_salarial_valor_medio").alias("salario_medio"),
        F.expr("percentile_approx(faixa_salarial_valor_medio, 0.5)").alias("salario_mediano"),
        F.count("*").alias("n_respondentes"),
    )
)
gravar_gold(gold_remuneracao, "gold_remuneracao")


## gold_diversidade_genero: Q3

In [ ]:
total_por_nivel = (
    respondentes.filter(F.col("nivel_senioridade").isNotNull())
    .groupBy("ano_pesquisa", "nivel_senioridade")
    .agg(F.count("*").alias("total_nivel"))
)
genero_por_nivel = (
    respondentes.filter(F.col("nivel_senioridade").isNotNull())
    .groupBy("ano_pesquisa", "nivel_senioridade", "genero")
    .agg(F.count("*").alias("n"))
)
gold_diversidade_genero = (
    genero_por_nivel.join(total_por_nivel, ["ano_pesquisa", "nivel_senioridade"])
    .withColumn("pct_dentro_do_nivel", F.round(F.col("n") / F.col("total_nivel") * 100, 1))
    .select("ano_pesquisa", "nivel_senioridade", "genero", "pct_dentro_do_nivel")
)
gravar_gold(gold_diversidade_genero, "gold_diversidade_genero")


## gold_tecnologias_adocao: Q4

In [ ]:
total_por_ano = respondentes.groupBy("ano_pesquisa").agg(F.count("*").alias("total_respondentes"))
tech_counts = (
    tecnologias.groupBy("ano_pesquisa", "categoria_tecnologia", "tecnologia")
    .agg(F.count("*").alias("n_respondentes"))
)
gold_tecnologias_adocao = (
    tech_counts.join(total_por_ano, "ano_pesquisa")
    .withColumn("pct_adocao", F.round(F.col("n_respondentes") / F.col("total_respondentes") * 100, 1))
    .drop("total_respondentes")
)
gravar_gold(gold_tecnologias_adocao, "gold_tecnologias_adocao")


## gold_adocao_ia e gold_prioridade_ia_empresa: Q5

In [ ]:
gold_adocao_ia = respondentes.groupBy("ano_pesquisa").agg(
    F.round(F.mean("usa_ia_generativa") * 100, 1).alias("pct_usa_ia_generativa"),
    F.round(F.mean("paga_por_ia_generativa") * 100, 1).alias("pct_paga_por_ia"),
    F.round(F.mean("usa_ia_para_codigo") * 100, 1).alias("pct_usa_ia_para_codigo"),
)
gravar_gold(gold_adocao_ia, "gold_adocao_ia")

base_gestores = respondentes.filter(F.col("prioridade_ia_empresa").isNotNull())
total_gestores = base_gestores.groupBy("ano_pesquisa").agg(F.count("*").alias("total_gestores"))
gold_prioridade_ia_empresa = (
    base_gestores.groupBy("ano_pesquisa", "prioridade_ia_empresa")
    .agg(F.count("*").alias("n_gestores"))
    .join(total_gestores, "ano_pesquisa")
    .withColumn("pct_gestores", F.round(F.col("n_gestores") / F.col("total_gestores") * 100, 1))
    .drop("total_gestores")
)
gravar_gold(gold_prioridade_ia_empresa, "gold_prioridade_ia_empresa")


## gold_modelo_trabalho: Q6

In [ ]:
gold_modelo_trabalho = (
    respondentes.filter(F.col("modelo_trabalho_atual").isNotNull())
    .groupBy("ano_pesquisa", "regiao", "modelo_trabalho_atual")
    .agg(F.count("*").alias("n_respondentes"))
)
gravar_gold(gold_modelo_trabalho, "gold_modelo_trabalho")


## Verificação final

In [ ]:
spark.sql(f"SHOW TABLES IN {DB_GOLD}").show(truncate=False)
